# motif-discover vs STREME: Live Benchmark

Runs both tools independently on the same ENCODE K562 ChIP-seq data and compares accuracy and speed.

## 1. Setup

In [ ]:
import os, subprocess

# Clone motif-discover repo
if not os.path.isdir('/content/motif-discover/.git'):
    subprocess.run('rm -rf /content/motif-discover && git clone -q https://github.com/Travis42/motif-discover.git /content/motif-discover', shell=True)
os.chdir('/content/motif-discover')
subprocess.run(['chmod', '+x', 'motif-discover'])

# Install STREME via apt (Colab runs Ubuntu)
if not os.path.exists('/usr/bin/streme'):
    print('Installing MEME Suite (STREME)...')
    subprocess.run('apt-get update -qq && apt-get install -y -qq meme-suite 2>&1 | tail -3', shell=True)

# Verify both binaries
r1 = subprocess.run(['./motif-discover', '--help'], capture_output=True, text=True, timeout=5)
r2 = subprocess.run(['/usr/bin/streme', '--version'], capture_output=True, text=True, timeout=5)
streme_ver = [l for l in r2.stdout.split('\n') if 'STREME' in l or 'version' in l.lower()]
n_tfs = len([f for f in os.listdir('example') if f.endswith('.fa')])
print(f'motif-discover: {os.path.getsize("motif-discover")//1024} KB (static binary)')
print(f'STREME:         {" ".join(streme_ver[:1]) if streme_ver else "installed"} (/usr/bin/streme)')
print(f'Example data:   {n_tfs} TFs')

## 2. Run motif-discover

In [ ]:
%%time
r_ours = subprocess.run(
    ['./motif-discover', '--data', 'example/', '--ours-only', '--no-meme',
     '--streme-bin', '/usr/bin/streme'],
    capture_output=True, text=True
)
print(r_ours.stdout)

## 3. Run STREME

Same data, same width range (6–17), same DNA alphabet.

In [ ]:
%%time
r_streme = subprocess.run(
    ['./motif-discover', '--data', 'example/', '--ours-only', '--no-meme',
     '--streme-bin', '/usr/bin/streme'],
    capture_output=True, text=True
)
print(r_streme.stdout)

## 4. Run Full Comparison

Our benchmark binary runs both tools on identical data and scores them with the same AUROC computation.

In [ ]:
import time

t0 = time.time()
r = subprocess.run(
    ['./motif-discover', '--data', 'example/', '--no-meme',
     '--streme-bin', '/usr/bin/streme'],
    capture_output=True, text=True
)
elapsed = time.time() - t0
print(f'Total wall time: {elapsed:.1f}s')
print(r.stdout)

## 5. Parse & Summarize

In [ ]:
import pandas as pd

rows = []
for line in r.stdout.strip().split('\n'):
    parts = line.split('\t')
    if len(parts) >= 5 and parts[0] != 'TF' and parts[4] in ('ours', 'streme'):
        rows.append({
            'TF': parts[0],
            'Width': int(parts[1]),
            'AUROC': float(parts[2]),
            'Time_s': float(parts[3]),
            'Tool': 'motif-discover' if parts[4] == 'ours' else 'STREME',
        })

df = pd.DataFrame(rows)
print(f'Parsed {len(df)} rows')

if len(df) > 0:
    ours = df[df['Tool'] == 'motif-discover'].set_index('TF')
    streme = df[df['Tool'] == 'STREME'].set_index('TF')
    common = sorted(ours.index.intersection(streme.index))
    
    print(f'\n{len(common)} TFs with both tools\n')
    print(f'motif-discover:  AUROC={ours.loc[common, "AUROC"].mean():.4f}  ({ours.loc[common, "Time_s"].mean():.2f}s/TF)')
    print(f'STREME:          AUROC={streme.loc[common, "AUROC"].mean():.4f}  ({streme.loc[common, "Time_s"].mean():.2f}s/TF)')
    print(f'Speedup:         {streme.loc[common, "Time_s"].mean() / ours.loc[common, "Time_s"].mean():.1f}x')
else:
    print('No data rows found. Stderr:')
    print(r.stderr[:500])

## 6. Visualize

In [ ]:
if len(df) > 0 and len(common) > 0:
    import matplotlib.pyplot as plt
    import numpy as np

    MD_COLOR = '#2166AC'
    ST_COLOR = '#D6604D'

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Panel A: Per-TF AUROC
    ax = axes[0]
    x = np.arange(len(common))
    w = 0.35
    ax.barh(x - w/2, ours.loc[common, 'AUROC'], w, color=MD_COLOR, alpha=0.8, label='motif-discover')
    ax.barh(x + w/2, streme.loc[common, 'AUROC'], w, color=ST_COLOR, alpha=0.8, label='STREME')
    ax.set_yticks(x)
    ax.set_yticklabels(common, fontsize=9)
    ax.set_xlabel('AUROC')
    ax.set_title('Per-TF AUROC')
    ax.axvline(0.5, color='gray', linestyle='--', alpha=0.3)
    ax.legend()

    # Panel B: Scatter
    ax = axes[1]
    o = ours.loc[common, 'AUROC'].values
    s = streme.loc[common, 'AUROC'].values
    colors = np.where(o > s, MD_COLOR, ST_COLOR)
    ax.scatter(s, o, alpha=0.7, s=60, c=colors)
    ax.plot([0.3, 1.0], [0.3, 1.0], 'k--', alpha=0.3)
    ax.set_xlabel('STREME AUROC')
    ax.set_ylabel('motif-discover AUROC')
    ax.set_title(f'Per-TF (n={len(common)})')
    for tf in common:
        d = ours.loc[tf, 'AUROC'] - streme.loc[tf, 'AUROC']
        if abs(d) > 0.1:
            ax.annotate(tf, (streme.loc[tf, 'AUROC'], ours.loc[tf, 'AUROC']), fontsize=8)

    # Panel C: Speed
    ax = axes[2]
    md_t = ours.loc[common, 'Time_s'].mean()
    st_t = streme.loc[common, 'Time_s'].mean()
    bars = ax.bar(['motif-discover', 'STREME'], [md_t, st_t],
                  color=[MD_COLOR, ST_COLOR], edgecolor='black', width=0.5)
    for bar, t in zip(bars, [md_t, st_t]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{t:.2f}s', ha='center', fontsize=12, fontweight='bold')
    ax.set_ylabel('Time per TF (seconds)')
    ax.set_title('Speed')

    plt.tight_layout()
    plt.show()
else:
    print('No comparable results to visualize')

## 7. Statistics

In [ ]:
if len(df) > 0 and len(common) > 0:
    diffs = ours.loc[common, 'AUROC'].values - streme.loc[common, 'AUROC'].values
    wins = (diffs > 0.001).sum()
    losses = (diffs < -0.001).sum()

    print(f'motif-discover wins: {wins}')
    print(f'STREME wins:         {losses}')
    print(f'Ties:                {len(diffs) - wins - losses}')
    print(f'Mean Δ AUROC:        {diffs.mean():+.4f}')

    if len(diffs) >= 5:
        from scipy.stats import wilcoxon
        _, p = wilcoxon(diffs)
        print(f'Wilcoxon p-value:    {p:.2e}')

---

Live benchmark on this machine. Full 132-TF results and paper: [github.com/Travis42/motif-discover](https://github.com/Travis42/motif-discover)